# PROJETO INTEGRADOR · 3º PERÍODO · INTELIGÊNCIA ARTIFICIAL
## Sistema Inteligente de Encaminhamento Social
### Avaliação Científica de Modelos de NLP, Validação Cruzada ($k=5$) e Análise da Base A (MDS)

**Integrantes do Grupo:** Paula Tomazzelli, Guilherme Henrique Carvalhal, Felippe Almeida, João Victor Domingues  
**Instituição:** UniEVANGÉLICA · Semestre 2026/2  
**Vínculo com ODS:** ODS 16 (Meta 16.6) e ODS 10 (Meta 10.2)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

# Configuração de estilo dos gráficos
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Análise Exploratória da Base A Real (MDS - CadÚnico Municipal)

In [ ]:
with open('../data/estatisticas_anapolis_mds.json', 'r', encoding='utf-8') as f:
    stats_anapolis = json.load(f)

print('=== ESTATÍSTICAS OFICIAIS DO CADÚNICO EM ANÁPOLIS - GO ===')
for k, v in stats_anapolis.items():
    print(f'{k}: {v}')

### 2. Carga dos Relatos Rotulados e Treinamento com Validação Cruzada ($k=5$)

In [ ]:
with open('../data/relatos_treino_social.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_relatos = pd.DataFrame(data)
print(f'Total de relatos: {len(df_relatos)}')
df_relatos['categoria'].value_counts().plot(kind='barh', color='teal', title='Distribuição de Classes nos Relatos')

### 3. Comparação Experimental: Linhas de Base vs Modelos Supervisionados

In [ ]:
X = df_relatos['texto'].values
y = df_relatos['categoria'].values

models = {
    'Multinomial Naive Bayes': Pipeline([('tfidf', TfidfVectorizer()), ('clf', MultinomialNB(alpha=0.5))]),
    'Linear SVM': Pipeline([('tfidf', TfidfVectorizer()), ('clf', LinearSVC(C=1.0, random_state=42))]),
    'Regressão Logística': Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression(C=2.0, max_iter=1000, random_state=42))])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_summary = {}

for name, model in models.items():
    res = cross_validate(model, X, y, cv=cv, scoring=['f1_macro', 'accuracy'])
    scores_summary[name] = {
        'F1-Macro Médio': np.mean(res['test_f1_macro']),
        'Desvio Padrão': np.std(res['test_f1_macro']),
        'Acurácia Média': np.mean(res['test_accuracy'])
    }

pd.DataFrame(scores_summary).T

### 4. Matriz de Confusão do Modelo Campeão (Linear SVM)

In [ ]:
best_model = models['Linear SVM']
best_model.fit(X, y)
y_pred = best_model.predict(X)

classes = sorted(list(set(y)))
cm = confusion_matrix(y, y_pred, labels=classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes)
plt.title('Matriz de Confusão - Linear SVM (100% Recall na Classe Violência)')
plt.xlabel('Predição')
plt.ylabel('Verdadeiro')
plt.show()